[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-bayesian-networks.ipynb)

# Bayesian Networks

*AIBits Academy · Machine Learning End To End · ⚠ Advanced Topic*

Naive Bayes (Chapter 22) assumes every feature is independent given the class — convenient, but usually false. Bayesian Networks keep the same conditional-probability machinery while letting you say exactly which variables depend on which.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

> **⚠ Why This Page Is Marked "Advanced"**
>
> This page assumes comfort with conditional probability and Bayes' theorem (Naive Bayes page) and builds toward exact probabilistic inference over a graph — different machinery from every predictive model elsewhere in this course, closer in spirit to the Bayesian Approach & Gaussian Processes and Monte Carlo/Markov Chain pages.

## Conditional Probability and Bayes' Theorem — A Refresher

For two events A and B, the conditional probability of A given B is P(A|B) = P(A,B) / P(B). Since the joint probability is symmetric — P(A,B) = P(B,A) — this gives Bayes' theorem:

$$P(A\mid B) = \dfrac{P(B\mid A)\cdot P(A)}{P(B)}$$

Naive Bayes uses this once per feature, under the simplifying assumption that every feature is conditionally independent of every other feature given the class. A **Bayesian Network** generalises this to an arbitrary set of variables with an arbitrary, explicitly-declared dependency structure — no blanket independence assumption required.

## From a Flat Assumption to a Graph

A Bayesian Network is a **Directed Acyclic Graph (DAG)**: each node is a random variable, and each edge represents a direct probabilistic dependency. Instead of one flat independence assumption across all features, you declare exactly which variables influence which — the graph structure *is* the model's assumptions, made explicit and inspectable. Every node stores a **Conditional Probability Table (CPT)**: the distribution of that node given each combination of its direct parents' values.

$$\text{Joint distribution factorises along the graph: } P(X_1,\dots,X_n) = \prod_i P(X_i \mid \mathrm{parents}(X_i))$$

This factorisation is what makes Bayesian Networks tractable — instead of needing a full joint probability table over all variables at once (which grows exponentially with the number of variables), you only ever need each node's small, local CPT.

## Worked Example — Festival-Season Demand at Zomato/Flipkart

A 3-node network modelling how festival season drives a retailer's sales spikes, with two parent influences on the sales node:

| CPT | Values |
|---|---|
| P(F=1) | 0.15 (festival days — Diwali, Holi, etc. — are ~15% of the year) |
| P(A=1 \| F=1) / P(A=1 \| F=0) | 0.90 / 0.30 (campaigns run far more often in festival season) |
| P(S=1 \| F,A) | F=1,A=1: 0.85 · F=1,A=0: 0.50 · F=0,A=1: 0.40 · F=0,A=0: 0.10 |

With these three CPTs, the full joint distribution over all 8 combinations of (F,A,S) is fully determined via the factorisation above — no need to specify it directly. **Exact inference** then means: given some evidence (say, "we observed a sales spike"), compute the posterior probability of the other, unobserved nodes by summing the joint distribution over every combination consistent with that evidence.

In [ ]:
from itertools import product
from fractions import Fraction as Fr

P_F = {1: Fr(15,100), 0: Fr(85,100)}
P_A_given_F = {1: Fr(90,100), 0: Fr(30,100)}
P_S_given_FA = {(1,1): Fr(85,100), (1,0): Fr(50,100),
                (0,1): Fr(40,100), (0,0): Fr(10,100)}

# Build the full joint table by the chain-rule factorisation: P(F,A,S) = P(F)·P(A|F)·P(S|F,A)
joint = {}
for f, a, s in product([0,1], repeat=3):
    p_a = P_A_given_F[f] if a==1 else 1-P_A_given_F[f]
    p_s = P_S_given_FA[(f,a)] if s==1 else 1-P_S_given_FA[(f,a)]
    joint[(f,a,s)] = P_F[f] * p_a * p_s

# Exact inference: P(Festival=1 | Sales Spike=1)
p_s1 = sum(v for (f,a,s),v in joint.items() if s==1)
p_f1_given_s1 = sum(v for (f,a,s),v in joint.items() if s==1 and f==1) / p_s1
print(f"P(Festival=1) prior:              {float(P_F[1]):.4f}")
print(f"P(Festival=1 | Sales Spike=1):    {float(p_f1_given_s1):.4f}")

Observing a sales spike alone nearly **triples** the probability estimate that it's festival season — from a 15% prior to a 43.08% posterior — purely from exact enumeration over the joint distribution, with no training data or optimisation involved. This is the core payoff of a Bayesian Network: given *any* subset of nodes as evidence, you can compute the posterior over any other subset, in any direction along the graph, not just the "predict Y from X" direction a normal classifier is restricted to.

## Explaining Away

A subtler query: given a sales spike, but knowing *no* ad campaign was running, does that make festival season more or less likely?

In [ ]:
denom = sum(v for (f,a,s),v in joint.items() if s==1 and a==0)
p_f1_given_s1_a0 = sum(v for (f,a,s),v in joint.items() if s==1 and a==0 and f==1) / denom
print(f"P(Festival=1 | Sales Spike=1, Ad Campaign=0):  {float(p_f1_given_s1_a0):.4f}")

0.1119 is *lower* than even the 0.15 prior — seeing a sales spike with no ad campaign running makes festival season *less* likely than having no information at all, not more. This is a real instance of **explaining away**: Ad Campaign=0 is itself already strong evidence against festival season (campaigns run 90% of the time in festival season but only 30% otherwise), and that pull dominates the weaker upward pull from the sales spike. Two parents of the same child node are marginally independent but become *dependent* once the child is observed — learning about one changes what you infer about the other, even though neither directly causes the other. Naive Bayes cannot represent this at all, since it assumes every feature stays independent of every other feature regardless of what's observed.

## Exact vs. Approximate Inference

The enumeration above is **exact inference** — summing the complete joint distribution. It's feasible here because the network has only 3 binary nodes (8 joint combinations). Real-world networks can have dozens of nodes, and the joint distribution's size grows exponentially with node count, making exact enumeration intractable. This is exactly where the **Monte Carlo simulation, Gibbs sampling, and Metropolis-Hastings** methods met on the earlier Advanced Extra Topics pages take over — sampling-based approximate inference that scales to networks far too large to enumerate exactly.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · The chain rule for one outcome

A retailer's network: Festival (F) → Ad campaign (A), and both → Sales spike (S). Using the probability tables below, compute `joint_111` = P(F=1, A=1, S=1) = P(F) · P(A|F) · P(S|F,A).

In [ ]:
from fractions import Fraction as Fr
P_F = {1: Fr(15, 100), 0: Fr(85, 100)}
P_A1_given_F = {1: Fr(90, 100), 0: Fr(30, 100)}   # P(A=1 | F)
P_S1_given_FA = {(1, 1): Fr(85, 100), (1, 0): Fr(50, 100), (0, 1): Fr(40, 100), (0, 0): Fr(10, 100)}   # P(S=1 | F, A)
joint_111 = None   # TODO


In [ ]:
try:
    check("0.15 * 0.9 * 0.85", joint_111 == Fr(11475, 100000))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from fractions import Fraction as Fr
P_F = {1: Fr(15, 100), 0: Fr(85, 100)}
P_A1_given_F = {1: Fr(90, 100), 0: Fr(30, 100)}
P_S1_given_FA = {(1, 1): Fr(85, 100), (1, 0): Fr(50, 100), (0, 1): Fr(40, 100), (0, 0): Fr(10, 100)}
joint_111 = P_F[1] * P_A1_given_F[1] * P_S1_given_FA[(1, 1)]

```

</details>

### Exercise 2 · Medium · Marginalise to get P(S=1)

Sum the joint probability over every combination of F and A to get `p_s1` = P(S=1).

In [ ]:
p_s1 = None   # TODO (reuse the tables)


In [ ]:
try:
    check("P(S=1) = 0.28375", p_s1 == Fr(28375, 100000))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
p_s1 = 0
for f in (0, 1):
    for a in (0, 1):
        p_a = P_A1_given_F[f] if a == 1 else 1 - P_A1_given_F[f]
        p_s1 += P_F[f] * p_a * P_S1_given_FA[(f, a)]

```

</details>

### Exercise 3 · Stretch · Reason backwards: was it a festival?

Sales spiked. Use Bayes' rule on the network to compute `post_festival` = P(F=1 | S=1) = P(F=1, S=1) / P(S=1), where P(F=1, S=1) sums over A.

In [ ]:
post_festival = None   # TODO


In [ ]:
try:
    check("posterior", abs(float(post_festival) - 0.4308) < 1e-3)
    check("higher than the 15% prior", post_festival > P_F[1])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
p_f1_s1 = 0
for a in (0, 1):
    p_a = P_A1_given_F[1] if a == 1 else 1 - P_A1_given_F[1]
    p_f1_s1 += P_F[1] * p_a * P_S1_given_FA[(1, a)]
post_festival = p_f1_s1 / p_s1

```

Seeing the spike raises the belief in a festival from 15% to about 43% — evidence flows backwards through the network.

</details>

---
*Back to the course: **Machine Learning End To End → Bayesian Networks**.*